In [ ]:
import requests
import keyring
import urllib3
from pyodata import Client

# ===================== 🔐 Credential Management =====================
# Before running this script for the first time, uncomment the two lines below to save your password.
# Re-comment them after running once to keep your credentials secure.
# pwd_to_save = "Your_SAP_Password"
# keyring.set_password("sap_100", "sap_username", pwd_to_save)
# ===================================================================

# 1. Basic Configuration
SERVICE_URL = 'https://yousap/sap/opu/odata/sap/ZMTEST5_UMD_SRV/'
SAP_USER = "sap_username"
SERVICE_NAME = "sap_100"

# Disable SSL certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 2. Securely Retrieve Password
pwd = keyring.get_password(SERVICE_NAME, SAP_USER)

if not pwd:
    raise ValueError(f"Password not found for service '{SERVICE_NAME}' and user '{SAP_USER}'. "
                     f"Please use keyring.set_password to store the password first.")

# 3. Create Session
session = requests.Session()
session.auth = (SAP_USER, pwd)
session.verify = False  # Common for Dev environments; use certificates for Production
session.timeout = 10

try:
    # 1. Initialize Client
    sap_client = Client(SERVICE_URL, session)
    
    # 2. Access Entity Set (scarrSet is the standard SAP Airline example table)
    scarr_set = sap_client.entity_sets.scarrSet

    # 3. Execute Query
    scarr_entities = scarr_set.get_entities().execute()
    
    # ===================== 🔥 Dynamic Output Section =====================
    fields = [
        ("Airline Code", "Carrid"),
        ("Airline Name", "Carrname"),
        ("Local Currency", "Currcode"),
        ("Airline URL", "Url")
    ]

    count = len(scarr_entities)
    print(f"=== Dynamic Field Output ({count} records found) ===\n")
    
    if count == 0:
        print("No matching data found.")
    else:
        for scarr_entity in scarr_entities:
            print("-" * 40)
            for label, field_name in fields:
                # getattr allows dynamic attribute access using string field names
                value = getattr(scarr_entity, field_name, "N/A") 
                print(f"{label:25}: {value}")

except Exception as e:
    print(f"An error occurred: {e}")